# 12 — Election data ward processing

This notebook derives ward/division-level summaries from the candidate-level file created in Notebook 11:

`local_election_results_raw_v1.csv`

It also reads the HoC ward-level tabs to add electorate, turnout, ballots, invalid votes and valid vote totals where available.

Outputs:

- `local_election_ward_source_v1.csv`
- `party_vote_totals_by_ward_v1.csv`
- `ward_result_summary_v1.csv`

## 12.1 Project paths and expected files

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

RAW_ELECTION_DIR = PROJECT_DIR / "data" / "raw" / "election_results"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "election_results"
DICTIONARY_DIR = PROJECT_DIR / "data" / "dictionaries"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_PATH = PROCESSED_DIR / "local_election_results_raw_v1.csv"

SOURCE_FILES = {
    2021: {
        "filename": "local_elections_2021_results-2.xlsx",
        "ward_sheet": "Wards-results",
        "ward_header": 1,
    },
    2022: {
        "filename": "local-elections-2022.xlsx",
        "ward_sheet": "Wards-results",
        "ward_header": 1,
    },
    2023: {
        "filename": "LEH-Candidates-2023.xlsx",
        "ward_sheet": "Ward_Level",
        "ward_header": 0,
    },
    2024: {
        "filename": "LEH-2024-results-HoC-version.xlsx",
        "ward_sheet": "Wards results",
        "ward_header": 1,
    },
    2025: {
        "filename": "LEH-2025-results-HoC.xlsx",
        "ward_sheet": "Ward results",
        "ward_header": 1,
    },
}

ELECTION_DATES = {
    2021: "2021-05-06",
    2022: "2022-05-05",
    2023: "2023-05-04",
    2024: "2024-05-02",
    2025: "2025-05-01",
}

if not CANDIDATE_PATH.exists():
    raise FileNotFoundError(f"Candidate file not found: {CANDIDATE_PATH}. Run Notebook 11 first.")

print("Project directory:", PROJECT_DIR)
print("Candidate file:", CANDIDATE_PATH)

Project directory: c:\Users\keena\Documents\Electoral_Tribes
Candidate file: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\local_election_results_raw_v1.csv


## 12.2 Helper functions

These duplicate key helper functions from Notebook 11 so this notebook can run independently.

In [2]:
def clean_colname(col):
    return str(col).strip()


def clean_str(value):
    if pd.isna(value):
        return pd.NA
    return str(value).strip()


def norm_key(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().upper()
    value = value.replace("&", "AND")
    value = re.sub(r"[^A-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def classify_geography_type(code):
    if pd.isna(code) or str(code).strip() == "":
        return "name_only_no_ons_code"

    code = str(code).strip().upper()

    if code.startswith("E58") or code.startswith("W58"):
        return "county_electoral_division"

    if code.startswith("E05") or code.startswith("W05"):
        return "electoral_ward_or_division"

    return "unknown_code_type"


def atlas_join_strategy(code, boundary_year):
    geography_type = classify_geography_type(code)

    if geography_type == "county_electoral_division":
        return "needs_county_electoral_division_geography"

    if geography_type == "electoral_ward_or_division" and int(boundary_year) == 2025:
        return "wd25_direct_candidate"

    if geography_type == "electoral_ward_or_division":
        return "needs_historical_ward_crosswalk"

    if geography_type == "name_only_no_ons_code":
        return "needs_ward_name_matching"

    return "manual_review"


def make_result_area_key(source_year, council_name, ward_name, ward_code):
    council_key = norm_key(council_name)
    ward_key = norm_key(ward_name)

    if pd.notna(ward_code) and str(ward_code).strip():
        return f"{source_year}|CODE|{council_key}|{str(ward_code).strip().upper()}|{ward_key}"

    return f"{source_year}|NAME|{council_key}|{ward_key}"


def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    return np.where(denominator > 0, numerator / denominator, np.nan)

## 12.3 Read and normalise HoC ward-level tabs

The ward-level tabs are used for electorate, turnout and total vote metadata. Party vote shares are calculated from the candidate file using the HoC `votes_effective` flag where available.

In [3]:
def normalise_ward_sheet(year, spec):
    path = RAW_ELECTION_DIR / spec["filename"]

    df = pd.read_excel(
        path,
        sheet_name=spec["ward_sheet"],
        header=spec["ward_header"],
    )

    df.columns = [clean_colname(c) for c in df.columns]
    df = df.dropna(how="all").copy()

    if year == 2021:
        out = pd.DataFrame({
            "source_year": year,
            "election_year": year,
            "election_date": ELECTION_DATES[year],
            "source_file": path.name,
            "source_sheet": spec["ward_sheet"],

            "county_name": df.get("County name"),
            "council_name": df["Local authority name"],
            "lad_code": df["Local authority code"],

            "ward_name": df["Ward/ED name"],
            "ward_code": df["Ward/ED code"],
            "ec_ward_code": pd.NA,
            "boundary_year": 2021,

            "election_type": df["Type"],
            "seats_available": df["Vacancies"],
            "electorate": df["Electorate"],
            "turnout": df["Turnout (%)"],
            "valid_votes": df["Total votes"],
            "ballots": pd.NA,
            "invalid_votes": pd.NA,
        })

    elif year == 2022:
        county_name = (
            df["COUNTYNAME"].combine_first(df["County name"])
            if "COUNTYNAME" in df.columns and "County name" in df.columns
            else df.get("COUNTYNAME", df.get("County name"))
        )

        out = pd.DataFrame({
            "source_year": year,
            "election_year": year,
            "election_date": ELECTION_DATES[year],
            "source_file": path.name,
            "source_sheet": spec["ward_sheet"],

            "county_name": county_name,
            "council_name": df["Local authority name"],
            "lad_code": df["Local authority code"],

            "ward_name": df["Ward name"],
            "ward_code": df["Ward code"],
            "ec_ward_code": pd.NA,
            "boundary_year": 2022,

            "election_type": df["Type"],
            "seats_available": df["Vacancies"],
            "electorate": df["Electorate"],
            "turnout": df["Turnout (%)"],
            "valid_votes": df["Total votes"],
            "ballots": pd.NA,
            "invalid_votes": pd.NA,
        })

    elif year == 2023:
        out = pd.DataFrame({
            "source_year": year,
            "election_year": year,
            "election_date": ELECTION_DATES[year],
            "source_file": path.name,
            "source_sheet": spec["ward_sheet"],

            "county_name": df["COUNTYNAME"],
            "council_name": df["DISTRICTNAME"],
            "lad_code": pd.NA,

            "ward_name": df["WARDNAME"],
            "ward_code": pd.NA,
            "ec_ward_code": pd.NA,
            "boundary_year": 2023,

            "election_type": df["TYPE"],
            "seats_available": df["VACS"],
            "electorate": df["ELECT"],
            "turnout": df["TURNOUT"],
            "valid_votes": df["Grand Total"],
            "ballots": pd.NA,
            "invalid_votes": pd.NA,
        })

    elif year == 2024:
        out = pd.DataFrame({
            "source_year": year,
            "election_year": year,
            "election_date": ELECTION_DATES[year],
            "source_file": path.name,
            "source_sheet": spec["ward_sheet"],

            "county_name": pd.NA,
            "council_name": df["Local authority name"],
            "lad_code": df["Local authority code"],

            "ward_name": df["Ward name"],
            "ward_code": df["Ward code"],
            "ec_ward_code": pd.NA,
            "boundary_year": 2024,

            "election_type": df["Election type"],
            "seats_available": df["Vacancies"],
            "electorate": df["Electorate"],
            "turnout": df["Turnout (%)"],
            "valid_votes": df["Total votes"],
            "ballots": pd.NA,
            "invalid_votes": pd.NA,
        })

    elif year == 2025:
        valid_votes = pd.to_numeric(df["Ballots"], errors="coerce") - pd.to_numeric(df["Invalid votes"], errors="coerce")

        out = pd.DataFrame({
            "source_year": year,
            "election_year": year,
            "election_date": ELECTION_DATES[year],
            "source_file": path.name,
            "source_sheet": spec["ward_sheet"],

            "county_name": pd.NA,
            "council_name": df["Lower tier authority"],
            "lad_code": pd.NA,

            "ward_name": df["Ward/ County Electoral District name"],
            "ward_code": df["ONS ward code"],
            "ec_ward_code": df["EC ward code"],
            "boundary_year": 2025,

            "election_type": df["Election type"],
            "seats_available": df["Seats"],
            "electorate": df["Electorate"],
            "turnout": df["Valid vote turnout (HoC method)"],
            "valid_votes": valid_votes,
            "ballots": df["Ballots"],
            "invalid_votes": df["Invalid votes"],
        })

    else:
        raise ValueError(f"No ward normaliser defined for {year}")

    for col in ["county_name", "council_name", "lad_code", "ward_name", "ward_code", "ec_ward_code", "election_type"]:
        if col in out.columns:
            out[col] = out[col].map(clean_str)

    for col in ["seats_available", "electorate", "turnout", "valid_votes", "ballots", "invalid_votes"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out["geography_type"] = out["ward_code"].map(classify_geography_type)

    out["atlas_join_strategy"] = [
        atlas_join_strategy(code, boundary_year)
        for code, boundary_year in zip(out["ward_code"], out["boundary_year"])
    ]

    out["result_area_key"] = [
        make_result_area_key(year, council, ward, code)
        for year, council, ward, code in zip(
            out["source_year"], out["council_name"], out["ward_name"], out["ward_code"]
        )
    ]

    return out

In [4]:
ward_source_frames = []

for year, spec in SOURCE_FILES.items():
    print(f"Processing ward sheet {year}...")
    year_df = normalise_ward_sheet(year, spec)
    ward_source_frames.append(year_df)

    print("  rows:", len(year_df))
    print("  result areas:", year_df["result_area_key"].nunique())
    print("  geography types:", year_df["geography_type"].value_counts(dropna=False).to_dict())

ward_source = pd.concat(ward_source_frames, ignore_index=True)

WARD_SOURCE_OUTPUT = PROCESSED_DIR / "local_election_ward_source_v1.csv"
ward_source.to_csv(WARD_SOURCE_OUTPUT, index=False)

print("Saved:", WARD_SOURCE_OUTPUT)
print("Rows:", len(ward_source))
print("Duplicate result_area_key rows:", ward_source["result_area_key"].duplicated().sum())

display(ward_source.head())

Processing ward sheet 2021...
  rows: 3863
  result areas: 3863
  geography types: {'electoral_ward_or_division': 2497, 'county_electoral_division': 1366}
Processing ward sheet 2022...
  rows: 3536
  result areas: 3536
  geography types: {'electoral_ward_or_division': 3535, 'name_only_no_ons_code': 1}
Processing ward sheet 2023...
  rows: 4797
  result areas: 4797
  geography types: {'name_only_no_ons_code': 4797}
Processing ward sheet 2024...
  rows: 1903
  result areas: 1903
  geography types: {'electoral_ward_or_division': 1903}
Processing ward sheet 2025...
  rows: 1401
  result areas: 1401
  geography types: {'county_electoral_division': 888, 'electoral_ward_or_division': 513}
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\local_election_ward_source_v1.csv
Rows: 15500
Duplicate result_area_key rows: 0


,source_year,election_year,election_date,source_file,source_sheet,county_name,council_name,lad_code,ward_name,ward_code,...,election_type,seats_available,electorate,turnout,valid_votes,ballots,invalid_votes,geography_type,atlas_join_strategy,result_area_key
0,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Wards-results,NaN,"Bristol, City Of",E06000023,Ashley,E05010885,...,UAA,3.0,14245.0,48.4,10098,NaN,NaN,electoral_ward_or_division,needs_historical_ward_crosswalk,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY
1,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Wards-results,NaN,"Bristol, City Of",E06000023,Avonmouth & Lawrence Weston,E05010886,...,UAA,3.0,15867.0,32.5,5131,NaN,NaN,electoral_ward_or_division,needs_historical_ward_crosswalk,2021|CODE|BRISTOL_CITY_OF|E05010886|AVONMOUTH_...
2,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Wards-results,NaN,"Bristol, City Of",E06000023,Bedminster,E05010887,...,UAA,2.0,9949.0,44.8,4365,NaN,NaN,electoral_ward_or_division,needs_historical_ward_crosswalk,2021|CODE|BRISTOL_CITY_OF|E05010887|BEDMINSTER
3,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Wards-results,NaN,"Bristol, City Of",E06000023,Bishopston & Ashley Down,E05010888,...,UAA,2.0,9430.0,55.7,5179,NaN,NaN,electoral_ward_or_division,needs_historical_ward_crosswalk,2021|CODE|BRISTOL_CITY_OF|E05010888|BISHOPSTON...
4,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Wards-results,NaN,"Bristol, City Of",E06000023,Bishopsworth,E05010889,...,UAA,2.0,9332.0,34.8,3124,NaN,NaN,electoral_ward_or_division,needs_historical_ward_crosswalk,2021|CODE|BRISTOL_CITY_OF|E05010889|BISHOPSWORTH


## 12.4 Load candidate-level file and mark effective party votes

For multi-member wards, the HoC datasets use `votes_effective` to identify the vote that should count toward party-level ward totals. If that flag is absent for an area, the notebook falls back to the highest-vote candidate for each party.

In [5]:
candidates = pd.read_csv(CANDIDATE_PATH, low_memory=False)

required_cols = ["result_area_key", "standard_party_label", "party_family", "votes", "elected", "votes_effective"]

missing = [c for c in required_cols if c not in candidates.columns]
if missing:
    raise ValueError(f"Candidate file missing required columns: {missing}")

candidates["votes"] = pd.to_numeric(candidates["votes"], errors="coerce").fillna(0)
candidates["votes_effective"] = candidates["votes_effective"].fillna(False).astype(bool)
candidates["elected"] = candidates["elected"].fillna(False).astype(bool)

# Start with HoC effective-vote flag.
candidates["effective_for_party_total"] = candidates["votes_effective"]

# If an area has no effective vote flags at all, fall back to highest-vote candidate per party.
has_effective = candidates.groupby("result_area_key")["effective_for_party_total"].transform("any")
fallback_idx = (
    candidates.loc[~has_effective]
    .groupby(["result_area_key", "standard_party_label"])["votes"]
    .idxmax()
    .dropna()
    .astype(int)
)

candidates.loc[fallback_idx, "effective_for_party_total"] = True

print("Candidate rows:", len(candidates))
print("Effective party vote rows:", candidates["effective_for_party_total"].sum())
display(candidates.head())

Candidate rows: 80392
Effective party vote rows: 61720


,result_id,result_area_key,source_year,election_date,election_year,election_type,ordinary_or_by_election,source_file,source_sheet,county_name,...,matched_wd25cd,matched_wd25nm,matched_lad25cd,matched_lad25nm,source_url,source_notes,data_quality_flag,manual_review_required,review_status,effective_for_party_total
0,364217414feb2f58,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY,2021,2021-05-06,2021,NaN,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,...,NaN,NaN,NaN,NaN,NaN,House of Commons Library Local Election Handbo...,ok,False,suggested,True
1,40f69ad91cd92fb8,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY,2021,2021-05-06,2021,NaN,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,...,NaN,NaN,NaN,NaN,NaN,House of Commons Library Local Election Handbo...,ok,False,suggested,True
2,cd14a3fbf2576559,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY,2021,2021-05-06,2021,NaN,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,...,NaN,NaN,NaN,NaN,NaN,House of Commons Library Local Election Handbo...,ok,False,suggested,False
3,0acd28cac5c5be06,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY,2021,2021-05-06,2021,NaN,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,...,NaN,NaN,NaN,NaN,NaN,House of Commons Library Local Election Handbo...,ok,False,suggested,False
4,6b848cda8054856f,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY,2021,2021-05-06,2021,NaN,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,...,NaN,NaN,NaN,NaN,NaN,House of Commons Library Local Election Handbo...,ok,False,suggested,False


## 12.5 Derive party vote totals by ward/division

In [6]:
effective = candidates[candidates["effective_for_party_total"]].copy()

party_votes = (
    effective
    .groupby(["result_area_key", "standard_party_label", "party_family"], as_index=False)
    .agg(
        party_votes=("votes", "sum")
    )
)

party_votes.to_csv(PROCESSED_DIR / "party_vote_totals_by_ward_v1.csv", index=False)

print("Saved party vote totals")
display(party_votes.head())

Saved party vote totals


,result_area_key,standard_party_label,party_family,party_votes
0,2021|CODE|ADUR|E05007562|BUCKINGHAM,Conservative,Conservative,646
1,2021|CODE|ADUR|E05007562|BUCKINGHAM,Green,Green,149
2,2021|CODE|ADUR|E05007562|BUCKINGHAM,Labour,Labour,364
3,2021|CODE|ADUR|E05007562|BUCKINGHAM,Liberal Democrat,Liberal Democrat,140
4,2021|CODE|ADUR|E05007563|CHURCHILL,Conservative,Conservative,590


## 12.6 Build core ward/division summary

In [7]:
meta_cols = [
    "source_year", "election_year", "election_date", "source_file",
    "council_name", "lad_code", "ward_name", "standard_ward_name",
    "ward_code", "ec_ward_code", "boundary_year", "geography_type",
    "atlas_join_strategy", "election_type", "ordinary_or_by_election",
    "seats_available"
]

meta = (
    candidates
    .groupby("result_area_key", as_index=False)
    .agg({c: "first" for c in meta_cols if c in candidates.columns})
)

candidate_count = (
    candidates
    .groupby("result_area_key", as_index=False)
    .agg(candidate_count=("candidate_name", "count"))
)

# Top two parties by effective party vote.
party_votes_sorted = party_votes.sort_values(
    ["result_area_key", "party_votes", "standard_party_label"],
    ascending=[True, False, True]
)

ranked = party_votes_sorted.groupby("result_area_key").head(2).copy()
ranked["party_rank"] = ranked.groupby("result_area_key").cumcount() + 1

top = (
    ranked
    .pivot(
        index="result_area_key",
        columns="party_rank",
        values=["standard_party_label", "party_votes"]
    )
    .reset_index()
)

top.columns = [
    "result_area_key",
    "top_party_by_votes",
    "runner_up_party_by_votes",
    "top_party_votes",
    "runner_up_party_votes",
]

# Ward-sheet metadata.
ward_meta = (
    ward_source
    .drop_duplicates("result_area_key", keep="first")
    [[
        "result_area_key", "electorate", "turnout", "valid_votes", "ballots", "invalid_votes", "seats_available"
    ]]
)

summary = (
    meta
    .merge(candidate_count, on="result_area_key", how="left")
    .merge(top, on="result_area_key", how="left")
    .merge(ward_meta, on="result_area_key", how="left", suffixes=("", "_ward"))
)

if "seats_available_ward" in summary.columns:
    summary["seats_available"] = summary["seats_available"].combine_first(summary["seats_available_ward"])
    summary = summary.drop(columns=["seats_available_ward"])

candidate_effective_total = (
    effective
    .groupby("result_area_key")["votes"]
    .sum()
    .rename("candidate_effective_total_votes")
)

summary = summary.merge(candidate_effective_total, on="result_area_key", how="left")

summary["valid_votes"] = summary["valid_votes"].combine_first(summary["candidate_effective_total_votes"])

summary["margin_votes"] = summary["top_party_votes"] - summary["runner_up_party_votes"]
summary["margin_pct"] = safe_divide(summary["margin_votes"], summary["valid_votes"])
summary["top_party_vote_share"] = safe_divide(summary["top_party_votes"], summary["valid_votes"])
summary["runner_up_vote_share"] = safe_divide(summary["runner_up_party_votes"], summary["valid_votes"])

display(summary.head())

,result_area_key,source_year,election_year,election_date,source_file,council_name,lad_code,ward_name,standard_ward_name,ward_code,...,electorate,turnout,valid_votes,ballots,invalid_votes,candidate_effective_total_votes,margin_votes,margin_pct,top_party_vote_share,runner_up_vote_share
0,2021|CODE|ADUR|E05007562|BUCKINGHAM,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Buckingham,Buckingham,E05007562,...,3106.0,42.1,1299.0,NaN,NaN,1299,282,0.217090,0.497306,0.280216
1,2021|CODE|ADUR|E05007563|CHURCHILL,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Churchill,Churchill,E05007563,...,3478.0,34.0,1187.0,NaN,NaN,1187,300,0.252738,0.497051,0.244313
2,2021|CODE|ADUR|E05007564|COKEHAM,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Cokeham,Cokeham,E05007564,...,3483.0,32.8,1130.0,NaN,NaN,1130,383,0.338938,0.608850,0.269912
3,2021|CODE|ADUR|E05007565|EASTBROOK,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Eastbrook,Eastbrook,E05007565,...,3471.0,36.1,1244.0,NaN,NaN,1244,94,0.075563,0.466238,0.390675
4,2021|CODE|ADUR|E05007566|HILLSIDE,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Hillside,Hillside,E05007566,...,3408.0,40.4,1129.0,NaN,NaN,1129,364,0.322409,0.621789,0.299380


## 12.7 Elected candidates and first loser details

In [8]:
# Candidate ranking by votes where no rank is available.
candidates["computed_rank"] = (
    candidates
    .sort_values(["result_area_key", "votes"], ascending=[True, False])
    .groupby("result_area_key")
    .cumcount() + 1
)

candidates["rank_for_summary"] = pd.to_numeric(candidates["rank"], errors="coerce").fillna(candidates["computed_rank"])

elected_rows = candidates[candidates["elected"]].copy()
losing_rows = candidates[~candidates["elected"]].copy()

elected_summary = (
    elected_rows
    .sort_values(["result_area_key", "votes"], ascending=[True, False])
    .groupby("result_area_key", as_index=False)
    .agg(
        elected_candidates=("candidate_name", lambda s: "; ".join(s.dropna().astype(str))),
        elected_parties=("standard_party_label", lambda s: "; ".join(s.dropna().astype(str))),
        lowest_elected_votes=("votes", "min")
    )
)

first_loser = (
    losing_rows
    .sort_values(["result_area_key", "votes"], ascending=[True, False])
    .groupby("result_area_key")
    .head(1)
    [[
        "result_area_key",
        "candidate_name",
        "standard_party_label",
        "votes"
    ]]
    .rename(columns={
        "candidate_name": "first_losing_candidate",
        "standard_party_label": "first_losing_party",
        "votes": "first_losing_votes",
    })
)

summary = (
    summary
    .merge(elected_summary, on="result_area_key", how="left")
    .merge(first_loser, on="result_area_key", how="left")
)

display(summary.head())

,result_area_key,source_year,election_year,election_date,source_file,council_name,lad_code,ward_name,standard_ward_name,ward_code,...,margin_votes,margin_pct,top_party_vote_share,runner_up_vote_share,elected_candidates,elected_parties,lowest_elected_votes,first_losing_candidate,first_losing_party,first_losing_votes
0,2021|CODE|ADUR|E05007562|BUCKINGHAM,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Buckingham,Buckingham,E05007562,...,282,0.217090,0.497306,0.280216,Boram K.,Conservative,646,Bannister M.,Labour,364.0
1,2021|CODE|ADUR|E05007563|CHURCHILL,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Churchill,Churchill,E05007563,...,300,0.252738,0.497051,0.244313,Buxton M.; Neocleous S.,Conservative; Conservative,585,Knight S.,Labour,290.0
2,2021|CODE|ADUR|E05007564|COKEHAM,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Cokeham,Cokeham,E05007564,...,383,0.338938,0.608850,0.269912,Wilkinson R.,Conservative,688,Aulton R.,Labour,305.0
3,2021|CODE|ADUR|E05007565|EASTBROOK,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Eastbrook,Eastbrook,E05007565,...,94,0.075563,0.466238,0.390675,Funnell J.R.; O'Neal C.J.,Conservative; Labour,486,Flower D.,Labour,472.0
4,2021|CODE|ADUR|E05007566|HILLSIDE,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Hillside,Hillside,E05007566,...,364,0.322409,0.621789,0.299380,Dunn A.D.,Conservative,702,Giles C.A.,Labour,338.0


## 12.8 Party-family vote buckets and fragmentation

This collapses standard party labels into broad analytical buckets used later for political-opportunity scoring.

In [9]:
def party_family_bucket(label, family):
    label_norm = norm_key(label).replace("_", " ")
    family_norm = norm_key(family).replace("_", " ")

    if "CONSERVATIVE" in label_norm or family_norm == "CONSERVATIVE":
        return "con"
    if "LABOUR" in label_norm or family_norm == "LABOUR":
        return "lab"
    if "LIBERAL DEMOCRAT" in label_norm or family_norm == "LIBERAL DEMOCRAT":
        return "ld"
    if label_norm == "GREEN" or "GREEN PARTY" in label_norm or family_norm == "GREEN":
        return "green"
    if "REFORM" in label_norm or "UKIP" in label_norm or "BREXIT" in label_norm or "REFORM UKIP BREXIT" in family_norm:
        return "reform_ukip_brexit"
    if "SDP" in label_norm or family_norm == "SDP":
        return "sdp"
    if "INDEPENDENT" in label_norm or "IND" == label_norm or "LOCALIST" in family_norm or "RESIDENT" in label_norm:
        return "independent"
    return "other"


party_votes["party_bucket"] = [
    party_family_bucket(label, family)
    for label, family in zip(party_votes["standard_party_label"], party_votes["party_family"])
]

bucket_votes = (
    party_votes
    .groupby(["result_area_key", "party_bucket"], as_index=False)
    .agg(votes=("party_votes", "sum"))
)

bucket_wide = (
    bucket_votes
    .pivot_table(
        index="result_area_key",
        columns="party_bucket",
        values="votes",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

expected_buckets = ["con", "lab", "ld", "green", "reform_ukip_brexit", "independent", "sdp", "other"]

for bucket in expected_buckets:
    if bucket not in bucket_wide.columns:
        bucket_wide[bucket] = 0

bucket_wide = bucket_wide.rename(columns={bucket: f"{bucket}_votes" for bucket in expected_buckets})

summary = summary.merge(bucket_wide, on="result_area_key", how="left")

for bucket in expected_buckets:
    votes_col = f"{bucket}_votes"
    share_col = f"{bucket}_share"
    summary[votes_col] = summary[votes_col].fillna(0)
    summary[share_col] = safe_divide(summary[votes_col], summary["valid_votes"])

# Fragmentation uses standard party-label vote shares, not broad buckets.
party_votes_with_total = party_votes.merge(
    summary[["result_area_key", "valid_votes"]],
    on="result_area_key",
    how="left"
)

party_votes_with_total["party_share"] = safe_divide(
    party_votes_with_total["party_votes"],
    party_votes_with_total["valid_votes"]
)

fragmentation = (
    party_votes_with_total
    .assign(share_sq=lambda d: d["party_share"] ** 2)
    .groupby("result_area_key", as_index=False)
    .agg(
        party_fragmentation_index=("share_sq", lambda s: 1 - s.sum()),
        effective_number_of_parties=("share_sq", lambda s: 1 / s.sum() if s.sum() > 0 else np.nan)
    )
)

summary = summary.merge(fragmentation, on="result_area_key", how="left")

display(summary.head())

,result_area_key,source_year,election_year,election_date,source_file,council_name,lad_code,ward_name,standard_ward_name,ward_code,...,con_share,lab_share,ld_share,green_share,reform_ukip_brexit_share,independent_share,sdp_share,other_share,party_fragmentation_index,effective_number_of_parties
0,2021|CODE|ADUR|E05007562|BUCKINGHAM,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Buckingham,Buckingham,E05007562,...,0.497306,0.280216,0.107775,0.114704,0.0,0.0,0.0,0.0,0.649394,2.852204
1,2021|CODE|ADUR|E05007563|CHURCHILL,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Churchill,Churchill,E05007563,...,0.497051,0.244313,0.128896,0.129739,0.0,0.0,0.0,0.0,0.659804,2.939486
2,2021|CODE|ADUR|E05007564|COKEHAM,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Cokeham,Cokeham,E05007564,...,0.608850,0.269912,0.000000,0.121239,0.0,0.0,0.0,0.0,0.541751,2.182220
3,2021|CODE|ADUR|E05007565|EASTBROOK,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Eastbrook,Eastbrook,E05007565,...,0.466238,0.390675,0.000000,0.143087,0.0,0.0,0.0,0.0,0.609521,2.560958
4,2021|CODE|ADUR|E05007566|HILLSIDE,2021,2021,2021-05-06,local_elections_2021_results-2.xlsx,Adur,E07000223,Hillside,Hillside,E05007566,...,0.621789,0.299380,0.000000,0.078831,0.0,0.0,0.0,0.0,0.517536,2.072691


## 12.9 Final quality flags and save output

In [10]:
summary["data_quality_flag"] = "ok"

summary.loc[summary["ward_code"].isna(), "data_quality_flag"] = "needs_ward_code_match"
summary.loc[summary["geography_type"].eq("county_electoral_division"), "data_quality_flag"] = "county_electoral_division_not_ward"
summary.loc[summary["valid_votes"].isna(), "data_quality_flag"] = "missing_valid_votes"

summary["manual_review_required"] = (
    summary["ward_code"].isna()
    | summary["geography_type"].isin(["county_electoral_division", "unknown_code_type", "name_only_no_ons_code"])
    | summary["valid_votes"].isna()
)

preferred_order = [
    "result_area_key",
    "source_year",
    "election_year",
    "election_date",
    "source_file",

    "council_name",
    "lad_code",
    "ward_name",
    "standard_ward_name",
    "ward_code",
    "ec_ward_code",
    "boundary_year",
    "geography_type",
    "atlas_join_strategy",

    "election_type",
    "ordinary_or_by_election",
    "seats_available",
    "electorate",
    "turnout",
    "valid_votes",
    "ballots",
    "invalid_votes",
    "candidate_count",

    "top_party_by_votes",
    "runner_up_party_by_votes",
    "top_party_votes",
    "runner_up_party_votes",
    "margin_votes",
    "margin_pct",
    "top_party_vote_share",
    "runner_up_vote_share",

    "elected_parties",
    "elected_candidates",
    "lowest_elected_votes",
    "first_losing_candidate",
    "first_losing_party",
    "first_losing_votes",

    "con_votes", "lab_votes", "ld_votes", "green_votes",
    "reform_ukip_brexit_votes", "independent_votes", "sdp_votes", "other_votes",
    "con_share", "lab_share", "ld_share", "green_share",
    "reform_ukip_brexit_share", "independent_share", "sdp_share", "other_share",

    "party_fragmentation_index",
    "effective_number_of_parties",

    "data_quality_flag",
    "manual_review_required",
]

ordered = [c for c in preferred_order if c in summary.columns]
remaining = [c for c in summary.columns if c not in ordered]

summary = summary[ordered + remaining]

WARD_SUMMARY_OUTPUT = PROCESSED_DIR / "ward_result_summary_v1.csv"
summary.to_csv(WARD_SUMMARY_OUTPUT, index=False)

print("Saved:", WARD_SUMMARY_OUTPUT)
print("Rows:", len(summary))
print("Manual review rows:", summary["manual_review_required"].sum())

report = (
    summary
    .groupby(["source_year", "geography_type", "data_quality_flag"], as_index=False)
    .agg(
        result_areas=("result_area_key", "count"),
        total_valid_votes=("valid_votes", "sum")
    )
)

display(report)

report.to_csv(PROCESSED_DIR / "ward_result_summary_processing_report_v1.csv", index=False)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\ward_result_summary_v1.csv
Rows: 15609
Manual review rows: 7085


,source_year,geography_type,data_quality_flag,result_areas,total_valid_votes
0,2021,county_electoral_division,county_electoral_division_not_ward,1366,5217981.0
1,2021,electoral_ward_or_division,ok,2498,6575733.0
2,2022,electoral_ward_or_division,ok,3610,8658048.0
3,2023,name_only_no_ons_code,needs_ward_code_match,4831,9229609.0
4,2024,electoral_ward_or_division,ok,1903,6009927.0
5,2025,county_electoral_division,county_electoral_division_not_ward,888,3138213.0
6,2025,electoral_ward_or_division,ok,513,1042938.0


## 12.10 Next notebook

The next notebook should be:

`13_join_election_results_to_k7_atlas.ipynb`

That notebook should not blindly join all rows. It should only join rows where geography compatibility is clear, for example:

- direct WD25 rows where `atlas_join_strategy == "wd25_direct_candidate"`
- historical ward rows after you add the appropriate OA→ward-year lookup
- county electoral divisions only after you add a county-division geography/crosswalk layer